# Order Flow Imbalance Feature Pipeline

This notebook shows a full pipeline developed in compute_ofi.py to compute multi-level Order Flow Imbalance (OFI) features from limit order book (LOB) data. It includes:

1. Timestamp preprocessing
2. Per-event OFI computation for each depth level
3. Rolling cumulative OFI computation over a time window
4. Depth normalization
5. Integrated OFI computation via PCA

Output: for each timestamp, `ofi_00_cum` is the best-level OFI, `ofi_00_cum` to `ofi_09_cum` are multi-level OFI and `ofi_integrated` is the integrated OFI. Cross-asset OFI is not defined in the paper, but it should refer to the cross-impact predictor in the linear regression model in section 3.1.3 and 3.1.4.

### Note: corrected a typo in Section 2.1 of the paper about the definition of OF. See details in compute_ofi.py.

In [1]:
from compute_ofi import *

## Usage
Load the given LOB data and run the pipeline. 

In [2]:
pd.set_option("display.max_columns", 100)  
df_raw = pd.read_csv("first_25000_rows.csv")
df_processed, ofi_cum_cols = compute_all_ofi(df_raw, depth=10, h='1s')
df_processed

,ts_event,ts_recv,rtype,publisher_id,instrument_id,action,side,depth,price,size,flags,ts_in_delta,sequence,bid_px_00,ask_px_00,bid_sz_00,ask_sz_00,bid_ct_00,ask_ct_00,bid_px_01,ask_px_01,bid_sz_01,ask_sz_01,bid_ct_01,ask_ct_01,bid_px_02,ask_px_02,bid_sz_02,ask_sz_02,bid_ct_02,ask_ct_02,bid_px_03,ask_px_03,bid_sz_03,ask_sz_03,bid_ct_03,ask_ct_03,bid_px_04,ask_px_04,bid_sz_04,ask_sz_04,bid_ct_04,ask_ct_04,bid_px_05,ask_px_05,bid_sz_05,ask_sz_05,bid_ct_05,ask_ct_05,bid_px_06,ask_px_06,bid_sz_06,ask_sz_06,bid_ct_06,ask_ct_06,bid_px_07,ask_px_07,bid_sz_07,ask_sz_07,bid_ct_07,ask_ct_07,bid_px_08,ask_px_08,bid_sz_08,ask_sz_08,bid_ct_08,ask_ct_08,bid_px_09,ask_px_09,bid_sz_09,ask_sz_09,bid_ct_09,ask_ct_09,symbol,ofi_00_cum,ofi_01_cum,ofi_02_cum,ofi_03_cum,ofi_04_cum,ofi_05_cum,ofi_06_cum,ofi_07_cum,ofi_08_cum,ofi_09_cum,ofi_integrated
0,2024-10-21 11:54:29.221064336,2024-10-21 11:54:29.221230963,10,2,38,C,B,1,233.62,2,130,166627,11405847,233.67,233.74,139,200,1,1,233.53,233.75,10,1,1,1,233.52,233.77,200,8,1,1,233.48,233.78,28,150,1,2,233.45,233.79,15,29,1,1,233.40,233.80,110,25,2,1,233.39,233.94,10,200,1,1,233.31,233.98,110,44,2,3,233.30,234.00,100,155,1,7,233.25,234.13,55,400,2,1,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-10-21 11:54:29.223769812,2024-10-21 11:54:29.223936626,10,2,38,A,B,0,233.67,2,130,166814,11405848,233.67,233.74,141,200,2,1,233.53,233.75,10,1,1,1,233.52,233.77,200,8,1,1,233.48,233.78,28,150,1,2,233.45,233.79,15,29,1,1,233.40,233.80,110,25,2,1,233.39,233.94,10,200,1,1,233.31,233.98,110,44,2,3,233.30,234.00,100,155,1,7,233.25,234.13,55,400,2,1,AAPL,0.020101,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
2,2024-10-21 11:54:29.225030400,2024-10-21 11:54:29.225196809,10,2,38,A,B,0,233.67,3,130,166409,11405849,233.67,233.74,144,200,3,1,233.53,233.75,10,1,1,1,233.52,233.77,200,8,1,1,233.48,233.78,28,150,1,2,233.45,233.79,15,29,1,1,233.40,233.80,110,25,2,1,233.39,233.94,10,200,1,1,233.31,233.98,110,44,2,3,233.30,234.00,100,155,1,7,233.25,234.13,55,400,2,1,AAPL,0.050218,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
3,2024-10-21 11:54:29.712434212,2024-10-21 11:54:29.712600612,10,2,38,A,B,2,233.52,200,130,166400,11406449,233.67,233.74,144,200,3,1,233.53,233.75,10,1,1,1,233.52,233.77,400,8,2,1,233.48,233.78,28,150,1,2,233.45,233.79,15,29,1,1,233.40,233.80,110,25,2,1,233.39,233.94,10,200,1,1,233.31,233.98,110,44,2,3,233.30,234.00,100,155,1,7,233.25,234.13,55,400,2,1,AAPL,0.048972,0.000000,1.958864,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.048972
4,2024-10-21 11:54:29.764673165,2024-10-21 11:54:29.764839221,10,2,38,C,B,2,233.52,200,130,166056,11406501,233.67,233.74,144,200,3,1,233.53,233.75,10,1,1,1,233.52,233.77,200,8,1,1,233.48,233.78,28,150,1,2,233.45,233.79,15,29,1,1,233.40,233.80,110,25,2,1,233.39,233.94,10,200,1,1,233.31,233.98,110,44,2,3,233.30,234.00,100,155,1,7,233.25,234.13,55,400,2,1,AAPL,0.049203,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000345
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,2024-10-21 13:04:16.583527688,2024-10-21 13:04:16.583694069,10,2,38,A,B,2,233.46,200,130,166381,15565398,233.51,233.61,1,20,1,1,233.50,233.69,100,200,1,1,233.46,234.00,200,127,1,4,233.45,234.10,15,100,1,1,233.40,234.13,109,400,2,1,233.39,234.17,10,1,1,1,233.33,234.25,45,100,2,1,233.31,234.41,110,500,2,1,233.30,234.43,100,105,1,2,233.25,234.50,55,63,2,4,AAPL,0.144939,-0.009663,-0.966261,-1.932523,-0.144939,-1.053225,-0.096626,-0.434818,-1.062888,-0.966261,-0.727329
4996,2024-10-21 13:04:17.976461017,2024-10-21 13:04:17.976627074,10,2,38,A,A,1,233.69,200,130,166057,15566644,233.51,23

Statistics of OFIs:

In [3]:
df_processed.iloc[:, 74:].describe()

,ofi_00_cum,ofi_01_cum,ofi_02_cum,ofi_03_cum,ofi_04_cum,ofi_05_cum,ofi_06_cum,ofi_07_cum,ofi_08_cum,ofi_09_cum,ofi_integrated
count,4999.000000,4999.000000,4999.000000,4999.000000,4999.000000,4999.000000,4999.000000,4999.000000,4999.000000,4999.000000,4997.000000
mean,-0.091172,-0.298530,-0.085698,-0.252925,-0.102663,-0.140823,0.018294,0.062543,0.013102,-0.040872,-0.109833
std,1.438144,2.679396,2.778635,3.001404,2.917859,2.401194,2.076652,2.017553,2.212449,2.018353,1.996401
min,-9.646114,-11.720831,-11.878508,-13.252454,-12.097599,-12.889135,-7.921697,-9.798465,-8.966575,-9.318233,-8.501594
25%,-0.050204,-0.945552,-0.699526,-0.533191,-0.506133,-0.397949,-0.296760,-0.351316,-0.449210,-0.275980,-0.782411
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.010737,0.519235,0.690526,0.521569,0.244865,0.051596,0.325908,0.297637,0.345827,0.308106,0.691050
max,6.557377,9.897352,10.225434,11.858172,14.078826,11.915752,16.236031,16.269864,13.053423,9.240574,9.486388


#### Question
It seems that some rows of the order book are incorrect. For example, on row 5 the bid prices are not all shifted correctly.